<a href="https://colab.research.google.com/github/alscop/ESAA-26-1/blob/main/ESAA_OB_week04_1_Review.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 수상작 리뷰

# 수상작 소개

**따릉이 대여량 예측 AI 해커톤**  

(https://dacon.io/competitions/official/235837/codeshare/3724?page=1&dtype=vote)

목표: 서울의 일별 온도, 습도, 강수량 등 일기 예보 데이터를 통해 따릉이 대여량을 예측

설명: 서울의 2018-2021년 4년 동안의 날씨 데이터와 2018-2020년의 따릉이 대여량 데이터가 주어집니다.  
이 데이터를 이용해 2021년의 따릉이 대여량을 예측해보세요.  
주어진 데이터 이외의 데이터는 사용 금지!  


평가:  NMAE (Normalized Mean Absolute Error)

데이터: 2018년 4-6월, 2019년 4-6월, 2020년 4-6월 데이터로 2021년 데이터 예측

- 기상 데이터는 하루에 총 8번 3시간 간격으로 발표되는 기상단기예보(SHRT) 데이터를 1일 평균으로 변환한 데이터입니다.
- 2019년 6월 4일 까지 sky_condition (하늘 상태) 코드  : 맑음(1), 구름조금(2), 구름많음(3), 흐림(4)
- 2020년  sky_condition (하늘 상태) 코드  : 맑음(1), 구름많음(3), 흐림(4)
- precipitation_form (강수 형태) 코드 : 없음(0), 비(1), 진눈깨비(2), 눈(3), 소나기(4)
  - 원본 데이터에는 없음(0), 비(1),소나기(4)가 포함되어있었으며 진눈깨비(2)와 눈(3) 성분은 존재하지 않습니다.



---


## 코드 흐름 요약


- XGBoost 사용

### 코드 흐름

1. 라이브러리 임포트
- 데이터 핸들링 툴
- 데이터 시각화 툴
- 머신러닝 모델, 교차검증 툴

2. 데이터 로드
- dtype 확인
  - 날짜 데이터 object: 추후 datetime 으로 변경.
- NA 확인: 결측치 없음

> 기본 질문 도출
- 연도별 대여량 변화
- 날씨에 따른 대여량 변화
- 평일/주말 영향 : (도메인) 출퇴근시간 사용 비중 높음, 평일/주말 변수 별도 분리 필요



3. EDA

- 날짜 분석
  - `pd.to_datetime()`
  - '연'/'월_일'로 분리
  - 월_일에 따른 대여량 hue=연도별 선그래프로 시각화
  - 연도에 따라 사용량 증가 추세: 중앙값으로 검증.
  - 대여량 매우 낮은 날은 계속 낮음

- 변수간 관계 분석
  - 히트맵 상관분석
  - pairplot + hue=연도별  
  : 연도별 사용량 증가는 기후적 요인X, 사용량 자체의 증가. 연별 증가량이 유사할 것이라 가정.  

- 요일 분석(평일/주말)
  - 중앙값 이용
  - 주말에 중앙값 자체는 오히려 높음.
  - 평일/주말 변수 채택.  


4. 데이터 전처리

- 변수 선택/생성
  - 불필요 피처(시각화 변수) 제거
  - 의미 고려
    - 풍향 drop
    - 습도, 기온 -> 불쾌지수 파생변수 생성
    - 강우확률 drop, 실제 강수량 채택
    - 월 drop, 기온 변수가 대체
  - 파생변수 추가
    - 기후 악조건: 풍속*하늘 상태(수가 높을수록 날씨 나쁨) -> 자전거 타기 어려운 상황 표현.
    - 체감 추위: 최저온도/풍속. (불쾌지수는 더운 상황에서만 적용됨.)
    - 일교차: 체감온도 반영
  - 파생변수와 의미 겹치는 변수도 drop(습도)

5. 모델링

- 모델 후보
  - 단순선형회귀
  - 랜덤포레스트
  - XGBoost
  - LightGBM: 데이터 수 많지 않아 제외.
- 평가지표: NMAE

- train/test 데이터셋 분리 안 함.   
  - 각 일자별로 어떤 기후 특성을 가지고 있는지 알기 어려움  
  - 그래도 일자가 흐름에 따른 어느정도의 대여량 증가 추세가 보임
  - 분리시 학습 유의미할지 확신 없음
  - 동일 기간으로 train, test 예측해야됨  
  - -> **X,y만 분리함.**  
여러번의 시행착오가 차라리 낫겠다는 판단

i. **단순 선형 회귀**  
- 오차율 30% 결과 매우 별로임.  
- 데이터 특성상 선형적 회귀계수 유의성 없음  

ii. **XGBoost**
- 트리모델: 특정 기준으로 날씨 극단값 선제적으로 제외해나가며 낮은 값 할당해줄 것으로 기대됨.  
- 정확도 좋은 결과



[모델 성능 향상 전략]
 - CV: 최적 파라미터 찾기. + etimator 개수는 늘림(전체 데이터를 학습하여 아예 새로운 연도를 학습해줘야 하므로 충분한 학습기 준비시킴)
 - 트리 시각화
 - 피처중요도 확인: 상승분 반영해야하는 '연도' 부분이 학습 반영 낮음

[연도별 상승분 반영 전략 설정]
- 연도별 증가 비율 계산, 확인.
- XGB가 상승분 추세 학습 불가
- 최종 예측 값에 일괄적으로 1.2 곱함(직전 상승비율 반영)








## 새롭게 알게 된 내용 / 어려운 내용 / 배울 점 등 정리 및 주요 코드 3줄 이상 작성

- 기본적인 데이터 배경 지식 관련 EDA/전처리 전략 질문을 사전에 도출하고 검증해 나간 과정.
- EDA 과정에서 확인한 데이터 분포 및 특징을 모델링 과정에서 재점검하는 과정 유익함.  
  - 트리모델 사용 후 트리구조 시각화
  - 피처중요도에 주요 피처로 개입되었는지 확인.
  - 모델에 학습 불가하다면 최종 예측값에 임의 계산하는 방법도 있음!
-  피처 의미 고려하여 임의로 피처 drop 및 파생변수 재구성.   
  : 특히 자전거 탑승에 실질적 영향이 가는 기상 변수를 사전 선택, 의미 재구성하여 파생변수 생성.




In [3]:
# 불쾌지수 공식의 활용
# def get_discomfort(humid, min_t, max_t):
    # 전체적인 탑승의 경향성을 반영하기 위해 출퇴근 시간의 사용량이 많음에도 불구하고, 평균온도로 고려합니다.
    # temp = (min_t + max_t)/2
    # humid = humid / 100

    # discomfort = 1.8 * temp - 0.558 * (1 - humid) * (1.8*temp - 26) + 32
    # return discomfort

In [4]:
# 추운 정도 반영
# train_df['cold_measure'] = train_df['low_temp'] / train_df['wind_speed']
# test_df['cold_measure'] = test_df['low_temp'] / test_df['wind_speed']

# 일교차 반영
# 일교차를 반영하면, 체감온도를 반영할 수 있다.
# train_df['temp_diff'] = train_df['high_temp'] - train_df['low_temp']
# test_df['temp_diff'] = test_df['high_temp'] - test_df['low_temp']

- train/test 분리 포기하고 X,y만 분리하는 전략.  

---
앙상블 없이 XGBoost 단일모델 사용했음에도 성능 좋았음. 전처리 과정이 큰 도움이 된 것으로 보임.